In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.01', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44
1,IIT: Pct Change due to behavior,0.14,0.31,0.55,0.72,0.90,1.10,1.32,1.58,1.89,2.28,1.07,2.44
2,IIT: Pct Change due to macro,1.29,2.57,3.89,5.24,6.63,8.06,9.53,11.04,12.59,14.17,7.52,13.91
3,IIT: Overall Pct Change in taxes,-4.09,-2.72,-1.22,0.23,1.74,3.31,4.94,6.65,8.47,10.42,2.76,10.34
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,4.66,9.33,14.32,19.34,24.52,29.87,35.41,41.14,47.10,53.33,27.89,54.17
6,CIT: Pct Change due to macro,-2.85,-5.37,-7.91,-10.27,-12.55,-14.73,-16.82,-18.83,-20.74,-22.57,-14.02,-24.39
7,CIT: Overall Pct Change in taxes,1.67,3.45,5.28,7.08,8.90,10.74,12.62,14.57,16.59,18.73,9.96,16.57
8,All: Pct Change due to tax rates,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.12
9,All: Pct Change due to behavior,0.41,0.85,1.38,1.85,2.33,2.84,3.39,3.98,4.64,5.39,2.70,5.63


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.26,-0.28,-0.30,-0.30,-0.31,-0.33,-0.34,-0.35,-0.37,-0.38,-3.22
9,Rev Change Due to Behavior,0.02,0.05,0.08,0.11,0.14,0.18,0.23,0.27,0.33,0.40,1.82
10,Rev Change Due to Macro,0.05,0.11,0.18,0.25,0.32,0.40,0.50,0.59,0.70,0.81,3.91
11,Total Revenue Change,-0.19,-0.13,-0.05,0.04,0.13,0.24,0.36,0.49,0.64,0.81,2.34


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.23,-0.25,-0.27,-0.28,-0.29,-0.30,-0.31,-0.33,-0.34,-0.35,-2.97
1,Rev Change Due to Behavior,0.01,0.01,0.03,0.04,0.05,0.06,0.08,0.09,0.12,0.15,0.63
2,Rev Change Due to Macro,0.06,0.12,0.19,0.27,0.36,0.45,0.55,0.66,0.78,0.92,4.36
3,Total Revenue Change,-0.18,-0.13,-0.06,0.01,0.09,0.18,0.29,0.40,0.53,0.67,1.81


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.22,-0.25,-0.26,-0.27,-0.28,-0.29,-0.30,-0.32,-0.33,-0.34,-2.86
1,Rev Change Due to Behavior,0.01,0.01,0.03,0.04,0.05,0.06,0.07,0.09,0.11,0.14,0.61
2,Rev Change Due to Macro,0.05,0.12,0.18,0.26,0.34,0.43,0.53,0.64,0.76,0.89,4.21
3,Total Revenue Change,-0.17,-0.12,-0.06,0.01,0.09,0.18,0.28,0.39,0.51,0.66,1.76


In [21]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95
1,Rev Change Due to Behavior,0.01,0.01,0.02,0.03,0.04,0.06,0.07,0.09,0.11,0.13,0.57
2,Rev Change Due to Macro,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.01,0.02,0.06
3,Total Revenue Change,0.01,-0.28,-0.27,-0.27,-0.27,-0.27,-0.26,-0.25,-0.23,-0.21,-2.31


In [22]:
df_levels.to_csv('og_usa_result_w_tcja_prod_1.csv')